# Doklejka v2: wybrane metody z `14.zip` na danych `StudyHours / Grade`

Ten notebook jest **dodatkiem** do:
- `wyklad_analiza_1_study.ipynb`
- `wyklad_analiza_1_realworldexploration.ipynb`

Założenie: **bazujemy na tych samych danych studentów**, żeby nie zmieniać kontekstu.  
Każdy fragment ma:
1. krótki opis,
2. gotowy kod,
3. możliwie ten sam zbiór danych.

Tam gdzie mały zbiór jest zbyt ubogi (np. `hexbin`), dodaję **mały wariant syntetyczny** oparty na tych samych danych.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from timeit import timeit
from IPython.display import display

# Dane bazowe zgodne z głównym notebookiem
names = ['Dan', 'Joann', 'Pedro', 'Rosie', 'Ethan', 'Vicky', 'Frederic', 'Jimmie',
         'Rhonda', 'Giovanni', 'Francesca', 'Rajab', 'Naiyana', 'Kian', 'Jenny',
         'Jakeem', 'Helena', 'Ismat', 'Anila', 'Skye', 'Daniel', 'Aisha']

study_hours = [10.0, 11.5, 9.0, 16.0, 9.25, 1.0, 11.5, 9.0, 8.5, 14.5, 15.5,
               13.75, 9.0, 8.0, 15.5, 8.0, 9.0, 6.0, 10.0, 12.0, 12.5, 12.0]

grades = [50, 50, 47, 97, 49, 3, 52, 42, 26, 74, 82, 62, 37, 15, 70, 27, 36, 35, 48, 52, 63, 64]

df_students = pd.DataFrame({
    'Name': names,
    'StudyHours': study_hours,
    'Grade': grades
})

df_students['Passed'] = df_students['Grade'] >= 60
df_students['Pass'] = df_students['Passed']  # dla zgodności z wcześniejszym notebookiem

def show_distribution(var_data, title='Rozkład'):
    fig, ax = plt.subplots(2, 1, figsize=(10, 4))
    ax[0].hist(var_data, bins=10)
    ax[0].set_title(title)
    ax[0].axvline(var_data.mean(), color='cyan', linestyle='--', linewidth=2, label='mean')
    ax[0].axvline(var_data.median(), color='red', linestyle='--', linewidth=2, label='median')
    ax[0].legend()

    ax[1].boxplot(var_data, vert=False)
    ax[1].set_xlabel('wartość')
    plt.tight_layout()
    plt.show()

def show_density(var_data, title='Gęstość'):
    plt.figure(figsize=(10, 4))
    var_data.plot.density()
    plt.title(title)
    plt.axvline(var_data.mean(), color='cyan', linestyle='--', linewidth=2, label='mean')
    plt.axvline(var_data.median(), color='red', linestyle='--', linewidth=2, label='median')
    plt.legend()
    plt.show()

df_students.head()


,Name,StudyHours,Grade,Passed,Pass
0,Dan,10.00,50,False,False
1,Joann,11.50,50,False,False
2,Pedro,9.00,47,False,False
3,Rosie,16.00,97,True,True
4,Ethan,9.25,49,False,False


## 1. IPython i szybkie mierzenie czasu
Na żywo możesz pokazać `pd.cut?`, `df_students.query?` albo `df_students.head??`. W zapisanym notebooku sensowniejsze jest krótkie porównanie czasu prostych operacji.

In [ ]:

grades_arr = np.array(df_students['Grade'])
t_numpy = timeit(lambda: grades_arr * 2, number=10000)
t_list = timeit(lambda: [x * 2 for x in grades], number=10000)

print(f'NumPy  (10000 powtórzeń): {t_numpy:.4f} s')
print(f'Lista  (10000 powtórzeń): {t_list:.4f} s')
print(f'Przyspieszenie ~ {t_list / max(t_numpy, 1e-9):.1f}x')


In [ ]:
grades_list = list(df_students['Grade'])
grades_arr = np.array(df_students['Grade'])

%timeit [x * 2 for x in grades_list]
%timeit grades_arr * 2

In [2]:
big_list = grades_list * 10000
big_arr = np.array(big_list)

%timeit [x * 2 for x in big_list]
%timeit big_arr * 2

NameError: name 'grades_list' is not defined

## 2. NumPy: `shape`, `reshape`, `newaxis`, `stack`
To jest dobre uzupełnienie Twojej części o tensorach i wymiarach.

In [ ]:

grades_arr = np.array(df_students['Grade'])
study_arr = np.array(df_students['StudyHours'])

print('shape grades:', grades_arr.shape)
print('shape study :', study_arr.shape)

student_matrix = np.column_stack([study_arr, grades_arr])
print('column_stack shape:', student_matrix.shape)
print(student_matrix[:5])

print('\nreshape do kolumny:')
print(grades_arr.reshape(-1, 1)[:5])

print('\nnewaxis:')
print(grades_arr[np.newaxis, :].shape, grades_arr[:, np.newaxis].shape)

print('\nvstack / hstack:')
print(np.vstack([study_arr, grades_arr]).shape)
print(np.hstack([study_arr[:5], grades_arr[:5]]))


## 3. Ufuncs i agregacje
Pokazujemy operacje na całych tablicach: bez pętli, czytelnie i szybko.

In [ ]:

print('Średnia ocen:', grades_arr.mean())
print('Mediana ocen:', np.median(grades_arr))
print('Percentyle ocen 25/50/75:', np.percentile(grades_arr, [25, 50, 75]))

distance_from_mean = np.abs(grades_arr - grades_arr.mean())
print('\nOdległość każdej oceny od średniej:')
print(distance_from_mean.round(2))

print('\nPierwiastek z czasu nauki (ufunc):')
print(np.sqrt(study_arr).round(3))


## 4. Broadcasting i standaryzacja
To jest bardzo naturalne rozszerzenie Twojej części o porównywaniu kolumn i normalizacji.

In [ ]:

df_broadcast = df_students.copy()
df_broadcast['StudyHours_z'] = (df_broadcast['StudyHours'] - df_broadcast['StudyHours'].mean()) / df_broadcast['StudyHours'].std(ddof=1)
df_broadcast['Grade_z'] = (df_broadcast['Grade'] - df_broadcast['Grade'].mean()) / df_broadcast['Grade'].std(ddof=1)

df_broadcast[['Name', 'StudyHours', 'StudyHours_z', 'Grade', 'Grade_z']].head(10)


## 5. Maski boolowskie
To warto pokazać obok Twojego filtrowania warunkowego: maska jest obiektem, który można dalej liczyć i łączyć.

In [ ]:

mask_high_grade = df_students['Grade'] >= 60
mask_long_study = df_students['StudyHours'] >= 12
mask_both = mask_high_grade & mask_long_study

print('Ilu studentów zdało?:', mask_high_grade.sum())
print('Jaki to odsetek?:', mask_high_grade.mean())

df_students.loc[mask_both, ['Name', 'StudyHours', 'Grade', 'Passed']]


## 6. Sortowanie i top-N
To jest praktyczny odpowiednik `argsort`, `sort_values`, `nlargest`.

In [ ]:

grade_order = np.argsort(df_students['Grade'].to_numpy())
print('Indeksy po sortowaniu rosnącym:', grade_order)

print('\nTop 5 ocen:')
display(df_students.nlargest(5, 'Grade')[['Name', 'StudyHours', 'Grade']])

print('\nNajmniej czasu nauki:')
display(df_students.nsmallest(5, 'StudyHours')[['Name', 'StudyHours', 'Grade']])


## 7. `loc`, `iloc`, `query`, `eval`
To dobrze domyka temat pobierania danych z DataFrame i pokazuje dwa style pracy.

In [ ]:

print('loc po nazwie:')
display(df_students.loc[df_students['Name'] == 'Vicky'])

print('iloc po pozycji:')
display(df_students.iloc[0:5, [0, 1, 2]])

print('query:')
display(df_students.query('Grade >= 60 and StudyHours >= 12'))

df_eval = df_students.eval('HoursPerGrade = StudyHours / Grade')
df_eval[['Name', 'StudyHours', 'Grade', 'HoursPerGrade']].head()


## 8. Braki danych: `isnull`, `fillna`, `dropna`
Tutaj świadomie psujemy mały fragment danych, żeby pokazać pełny workflow naprawy.

In [ ]:

df_missing = df_students.copy()
df_missing.loc[[2, 7], 'StudyHours'] = np.nan
df_missing.loc[[5], 'Grade'] = np.nan

print('Braki w danych:')
display(df_missing.isnull().sum())

df_filled = df_missing.copy()
df_filled['StudyHours'] = df_filled['StudyHours'].fillna(df_filled['StudyHours'].mean())
df_filled['Grade'] = df_filled['Grade'].fillna(df_filled['Grade'].median())

print('Po fillna:')
display(df_filled.head(10))

print('Po dropna (usuń wiersze z brakami):')
display(df_missing.dropna())


## 9. `concat` i `merge`
To jest mały krok od jednej tabeli do łączenia kilku źródeł danych.

In [ ]:

df_meta = pd.DataFrame({
    'Name': names,
    'City': ['Kraków', 'Warszawa', 'Kraków', 'Wrocław', 'Poznań', 'Warszawa',
             'Gdańsk', 'Kraków', 'Łódź', 'Poznań', 'Warszawa', 'Kraków',
             'Wrocław', 'Gdańsk', 'Poznań', 'Łódź', 'Kraków', 'Warszawa',
             'Gdańsk', 'Wrocław', 'Poznań', 'Kraków'],
    'Track': ['A', 'B', 'A', 'A', 'B', 'C', 'A', 'B', 'C', 'A', 'B', 'A',
              'B', 'C', 'A', 'B', 'C', 'A', 'B', 'C', 'A', 'B']
})

extra_flag = pd.Series(df_students['StudyHours'] >= 12, name='LongStudy')

print('concat (doklejenie kolumny):')
display(pd.concat([df_students, extra_flag], axis=1).head())

print('merge (łączenie po Name):')
display(pd.merge(df_students, df_meta, on='Name', how='left').head(10))


## 10. `groupby().agg()`
To jest najważniejsze rozszerzenie Twojej części o `groupby`: kilka statystyk naraz, czytelnie i w jednej tabeli.

In [ ]:

group_summary = (
    df_students.groupby('Passed')
    .agg(
        n=('Name', 'size'),
        mean_grade=('Grade', 'mean'),
        median_grade=('Grade', 'median'),
        mean_hours=('StudyHours', 'mean'),
        max_grade=('Grade', 'max')
    )
    .reset_index()
)

group_summary


## 11. `transform()`
`agg()` daje jedną wartość na grupę, a `transform()` zwraca wynik w rozmiarze oryginalnej tabeli.

In [ ]:

df_transform = df_students.copy()
df_transform['Grade_vs_pass_group_mean'] = (
    df_transform['Grade'] - df_transform.groupby('Passed')['Grade'].transform('mean')
)

df_transform[['Name', 'Passed', 'Grade', 'Grade_vs_pass_group_mean']].head(10)


## 12. `cut` i `qcut`
To jest bardzo wygodne przed `groupby` i `pivot_table`: sami budujemy grupy przedziałowe.

In [ ]:

df_bands = df_students.copy()
df_bands['StudyBand'] = pd.cut(
    df_bands['StudyHours'],
    bins=[0, 8, 10, 12, 20],
    labels=['<=8', '8-10', '10-12', '12+']
)

df_bands['GradeQuartile'] = pd.qcut(
    df_bands['Grade'],
    q=4,
    duplicates='drop'
)

display(df_bands[['Name', 'StudyHours', 'StudyBand', 'Grade', 'GradeQuartile']].head(10))
display(df_bands['StudyBand'].value_counts().sort_index())


## 13. `pivot_table`
Na tych samych danych też da się sensownie pokazać pivot: np. `StudyBand × Passed`.

In [ ]:
df_bands = df_bands.copy()
df_bands['PassLabel'] = df_bands['Passed'].map({True: 'Pass', False: 'NoPass'})

pivot_mean = pd.pivot_table(
    df_bands,
    index='StudyBand',
    columns='PassLabel',
    values='Grade',
    aggfunc='mean',
    fill_value=0,
    observed=False,
    margins=True
)

pivot_count = pd.pivot_table(
    df_bands,
    index='StudyBand',
    columns='PassLabel',
    values='Grade',
    aggfunc='count',
    fill_value=0,
    observed=False,
    margins=True
)

print('Średnia ocena:')
display(pivot_mean.round(2))
print('Liczebności:')
display(pivot_count)

## 14. Stringi w Pandas
To będzie bardzo przydatne później przy ankiecie: czyszczenie napisów i standaryzacja odpowiedzi.

In [ ]:

df_names = df_students[['Name']].copy()
df_names['NameMessy'] = ['  ' + n.lower() + '  ' if i % 2 == 0 else n.upper() for i, n in enumerate(df_names['Name'])]

df_names['NameClean'] = (
    df_names['NameMessy']
    .str.strip()
    .str.title()
)

df_names['Contains_A'] = df_names['NameClean'].str.contains('a', case=False)
df_names.head(10)


## 15. Histogram, boxplot i gęstość + PDF
Krótka teoria: histogram pokazuje dane dyskretne w koszykach, a PDF to gładkie przybliżenie rozkładu ciągłego. Pole pod PDF ma wynosić 1.

In [ ]:

show_distribution(df_students['Grade'], title='Oceny: histogram + boxplot')
show_density(df_students['Grade'], title='Oceny: gęstość')

mu = df_students['Grade'].mean()
sigma = df_students['Grade'].std(ddof=1)
x = np.linspace(df_students['Grade'].min() - 10, df_students['Grade'].max() + 10, 400)
pdf = stats.norm.pdf(x, loc=mu, scale=sigma)

plt.figure(figsize=(10, 4))
plt.hist(df_students['Grade'], bins=8, density=True, alpha=0.6, label='Histogram (density=True)')
plt.plot(x, pdf, color='crimson', linewidth=2, label='PDF N(mean, std)')
plt.title('Histogram i przykładowa krzywa PDF')
plt.xlabel('Grade')
plt.ylabel('Gęstość')
plt.legend()
plt.show()

area = np.trapz(pdf, x)
within_one_sigma = ((df_students['Grade'] >= mu - sigma) & (df_students['Grade'] <= mu + sigma)).mean()
print(f'Przybliżone pole pod PDF: {area:.4f}')
print(f'Odsetek ocen w zakresie mean ± 1 std: {within_one_sigma:.2%}')


## 16. Error bars
To jest dobra mini-wstawka przy porównywaniu grup: nie tylko średnia, ale też rozrzut.

In [ ]:

band_stats = (
    df_bands.groupby('StudyBand', observed=False)
    .agg(mean_grade=('Grade', 'mean'), std_grade=('Grade', 'std'), n=('Grade', 'size'))
    .reset_index()
)

plt.figure(figsize=(8, 4))
plt.errorbar(
    band_stats['StudyBand'].astype(str),
    band_stats['mean_grade'],
    yerr=band_stats['std_grade'],
    fmt='o-',
    capsize=5
)
plt.title('Średnia ocena ± odchylenie standardowe')
plt.xlabel('StudyBand')
plt.ylabel('Grade')
plt.show()

band_stats.round(2)


## 17. Scatter, korelacja i regresja
To naturalne rozwinięcie Twojego `realworldexploration`.

In [ ]:

corr = df_students['StudyHours'].corr(df_students['Grade'])
m, b, r, p, se = stats.linregress(df_students['StudyHours'], df_students['Grade'])

x_line = np.linspace(df_students['StudyHours'].min(), df_students['StudyHours'].max(), 100)
y_line = m * x_line + b

plt.figure(figsize=(7, 4))
plt.scatter(df_students['StudyHours'], df_students['Grade'], c=df_students['Passed'].map({True: 'teal', False: 'orange'}), alpha=0.8)
plt.plot(x_line, y_line, color='crimson', linewidth=2)
plt.title(f'StudyHours vs Grade | corr={corr:.3f}')
plt.xlabel('StudyHours')
plt.ylabel('Grade')
plt.show()

print(f'm = {m:.4f}, b = {b:.4f}, r = {r:.4f}, p = {p:.4g}')


## 18. `subplots` i `annotate`
To jest wygodny sposób na pokazanie kilku widoków tych samych danych w jednym miejscu.

In [ ]:

top_student = df_students.loc[df_students['Grade'].idxmax()]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df_students['Grade'], bins=8, color='steelblue')
axes[0].set_title('Histogram ocen')

axes[1].boxplot(df_students['StudyHours'], vert=False)
axes[1].set_title('Boxplot StudyHours')

axes[2].scatter(df_students['StudyHours'], df_students['Grade'], color='gray', alpha=0.8)
axes[2].annotate(
    top_student['Name'],
    xy=(top_student['StudyHours'], top_student['Grade']),
    xytext=(top_student['StudyHours'] + 0.3, top_student['Grade'] - 5),
    arrowprops=dict(arrowstyle='->')
)
axes[2].set_title('Scatter z adnotacją')
axes[2].set_xlabel('StudyHours')
axes[2].set_ylabel('Grade')

plt.tight_layout()
plt.show()


## 19. `hexbin` jako opcjonalny bonus
Na małym zbiorze to nie ma sensu, więc generujemy większą próbkę syntetyczną wokół tych samych danych.

In [ ]:

rng = np.random.default_rng(42)
base = df_students[['StudyHours', 'Grade']].to_numpy()
rep = np.repeat(base, 25, axis=0)
noise = rng.normal(loc=0.0, scale=[0.35, 3.5], size=rep.shape)
big = pd.DataFrame(rep + noise, columns=['StudyHours', 'Grade'])

plt.figure(figsize=(7, 4))
plt.hexbin(big['StudyHours'], big['Grade'], gridsize=18, cmap='viridis')
plt.colorbar(label='liczba punktów w koszyku')
plt.title('Hexbin na większej próbie syntetycznej')
plt.xlabel('StudyHours')
plt.ylabel('Grade')
plt.show()


## 20. Mini-podsumowanie dydaktyczne
Najmocniejsze rzeczy do realnego wykorzystania później przy ankiecie i maratończykach to:
- maski boolowskie,
- `groupby().agg()`,
- `transform()`,
- `cut/qcut`,
- `pivot_table`,
- czyszczenie stringów,
- histogram + boxplot + scatter,
- regresja jako relacja, a nie tylko gotowa funkcja.

In [ ]:
df_students.head()